In [1]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This finds the files whether you're on Kaggle or on your own computer
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(f"**/{name}", recursive=True)
    return hits[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))

print("Weddings in our notebook (train):", len(train))
print("Weddings to predict (test):", len(test))
train.head()

Weddings in our notebook (train): 800
Weddings to predict (test): 400


,wedding_id,event,city,guests,aloo_count,mutton_kg,borhani_glasses,fairy_lights,drone_photographer,dhol_players,aunties_asking_when_marriage,went_back_for_seconds
0,W0001,Gaye Holud,Barishal,604,1327.0,138.1,529,478,no,6,8,0
1,W0002,Bou-bhat,Dhaka,166,104.0,35.0,172,445,yes,3,3,0
2,W0003,Biye,Mymensingh,67,59.0,16.7,72,116,yes,2,3,0
3,W0004,Biye,Dhaka,466,1206.0,122.3,668,1586,no,1,4,0
4,W0005,Biye,Khulna,287,209.0,81.9,278,646,yes,0,9,0


Each row is **one wedding**. The last column, `went_back_for_seconds`, is what we want to predict:
**1** = yes, guests went back for seconds and **0** = no.

The test file doesn't have this column. That's what *you* have to guess!

In [2]:
# How often do guests go back for seconds?
train["went_back_for_seconds"].value_counts()

went_back_for_seconds
0    464
1    336
Name: count, dtype: int64

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   wedding_id                    800 non-null    object 
 1   event                         800 non-null    object 
 2   city                          800 non-null    object 
 3   guests                        800 non-null    int64  
 4   aloo_count                    776 non-null    float64
 5   mutton_kg                     800 non-null    float64
 6   borhani_glasses               800 non-null    int64  
 7   fairy_lights                  800 non-null    object 
 8   drone_photographer            800 non-null    object 
 9   dhol_players                  800 non-null    int64  
 10  aunties_asking_when_marriage  800 non-null    int64  
 11  went_back_for_seconds         800 non-null    int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 75.1+ KB


In [4]:
# Fix 1: turn Bangla digits into English digits
numeric_cols=['guests','aloo_count','mutton_kg','borhani_glasses','fairy_lights','aunties_asking_when_marriage']

bangla_to_english = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def BN_to_EN(df, numeric_cols):
        for c in numeric_cols:
            df[c] = pd.to_numeric(
                df[c].astype(str).str.translate(bangla_to_english),errors="coerce",)
        return df
    
    
    #df["fairy_lights"] = df["fairy_lights"].astype(str).str.translate(bangla_to_english).astype(int)

train = BN_to_EN(train, numeric_cols)
test = BN_to_EN(test, numeric_cols)

print("Fixed!")

Fixed!


In [5]:
# Fix 2: how many empty boxes are there?
train.isna().sum()

wedding_id                       0
event                            0
city                             0
guests                           0
aloo_count                      24
mutton_kg                        0
borhani_glasses                  0
fairy_lights                     0
drone_photographer               0
dhol_players                     0
aunties_asking_when_marriage     0
went_back_for_seconds            0
dtype: int64

**Fix 3 is your mission.** Somewhere in `aloo_count` there are a few numbers that are 10 times too big.
Try `train.describe()` or sort the column with `train.sort_values("aloo_count")`. Can you spot Tutul's extra zeros?

In [6]:
train.describe()

,guests,aloo_count,mutton_kg,borhani_glasses,fairy_lights,dhol_players,aunties_asking_when_marriage,went_back_for_seconds
count,800.000000,776.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000
mean,371.623750,626.070876,94.350000,428.318750,970.915000,4.005000,5.996250,0.420000
std,184.408379,781.629417,49.023904,230.407713,600.886519,2.575513,2.491786,0.493867
min,51.000000,28.000000,9.300000,45.000000,81.000000,0.000000,1.000000,0.000000
25%,219.500000,262.750000,54.400000,241.000000,512.750000,2.000000,4.000000,0.000000
50%,367.000000,471.500000,92.900000,411.000000,821.500000,4.000000,6.000000,0.000000
75%,531.000000,850.000000,128.425000,597.000000,1305.500000,6.000000,8.000000,1.000000
max,700.000000,11520.000000,221.300000,1016.000000,2861.000000,8.000000,16.000000,1.000000


In [7]:
print(train["aloo_count"].sort_values(ascending=False).head(20))

501    11520.0
785    10270.0
671     8320.0
80      7150.0
447     6520.0
674     1831.0
556     1764.0
744     1739.0
677     1732.0
545     1712.0
196     1710.0
459     1684.0
13      1677.0
148     1665.0
678     1650.0
679     1602.0
641     1584.0
643     1553.0
289     1540.0
733     1537.0
Name: aloo_count, dtype: float64


In [8]:
print((test["aloo_count"] / test["guests"]).describe())
print((train["aloo_count"] / train["guests"]).describe())

count    400.000000
mean       1.529407
std        0.640855
min        0.402844
25%        0.953242
50%        1.535145
75%        2.069487
max        2.684466
dtype: float64
count    776.000000
mean       1.671515
std        1.764589
min        0.406015
25%        0.957247
50%        1.507989
75%        2.132390
max       26.752412
dtype: float64


In [9]:
train["ratio"] = train["aloo_count"] / train["guests"]
print(train["ratio"].sort_values(ascending=False).head(10))

671    26.752412
785    23.394077
80     22.698413
501    16.575540
447    16.464646
316     9.699248
115     2.700000
519     2.699758
28      2.698413
217     2.697168
Name: ratio, dtype: float64


In [10]:
# fix 10times bigger num

def alo_cleaning(df,max_ratio=3):
    df = df.copy()
    df["aloo_count"] = df["aloo_count"].where(df["ratio"] <= max_ratio, df["aloo_count"]/10)
    
    return df

train = alo_cleaning(train)

train = train.drop(columns="ratio", errors="ignore")

In [11]:
print((test["aloo_count"] / test["guests"]).describe())
print((train["aloo_count"] / train["guests"]).describe())

count    400.000000
mean       1.529407
std        0.640855
min        0.402844
25%        0.953242
50%        1.535145
75%        2.069487
max        2.684466
dtype: float64
count    776.000000
mean       1.537461
std        0.674965
min        0.406015
25%        0.957247
50%        1.506818
75%        2.127857
max        2.700000
dtype: float64


# Missing Values

median_ratio = (train["aloo_count"] / train["guests"]).median()

def Missing_values(df,median_ratio):
    miss = df["aloo_count"].isna()
    df.loc[miss, "aloo_count"] = (df.loc[miss, "guests"] * median_ratio).round()
    
    return df

train = Missing_values(train,median_ratio)
test = Missing_values(test,median_ratio)

print("Fixed")

In [12]:
train = train.dropna(subset=["aloo_count"]).reset_index(drop=True)
print(len(train))

776


In [13]:
train.head(5)

,wedding_id,event,city,guests,aloo_count,mutton_kg,borhani_glasses,fairy_lights,drone_photographer,dhol_players,aunties_asking_when_marriage,went_back_for_seconds
0,W0001,Gaye Holud,Barishal,604,1327.0,138.1,529,478,no,6,8,0
1,W0002,Bou-bhat,Dhaka,166,104.0,35.0,172,445,yes,3,3,0
2,W0003,Biye,Mymensingh,67,59.0,16.7,72,116,yes,2,3,0
3,W0004,Biye,Dhaka,466,1206.0,122.3,668,1586,no,1,4,0
4,W0005,Biye,Khulna,287,209.0,81.9,278,646,yes,0,9,0


If you found the secret, your tiny tree should beat Rafi's hundreds of trees, and you can read the rule it found like a sentence.

Can you write the rule in your own words? Put it in your notebook! The best explanation wins the **Best Aloo Theory** prize.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [15]:
X = train.drop(columns=['went_back_for_seconds'])
y = train['went_back_for_seconds']

#split the raw data
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
num_cols = X_tr.select_dtypes(include="number").columns.tolist()
cat_cols = X_tr.select_dtypes(exclude="number").columns.tolist()

print(num_cols)
print(cat_cols)

['guests', 'aloo_count', 'mutton_kg', 'borhani_glasses', 'fairy_lights', 'dhol_players', 'aunties_asking_when_marriage']
['wedding_id', 'event', 'city', 'drone_photographer']


In [17]:
#preprocessing + model
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

model = Pipeline([
    ("pre", pre),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=42)),
])

model.fit(X_tr, y_tr)

#permutation importance on the validation set
result = permutation_importance(
    model, X_val, y_val, n_repeats=10, random_state=42, scoring="f1_macro"
)

perm = pd.Series(result.importances_mean, index=X_val.columns).sort_values(ascending=False)
print(perm)

aloo_count                      0.153584
mutton_kg                       0.028891
guests                          0.019644
borhani_glasses                 0.016947
aunties_asking_when_marriage    0.010121
wedding_id                      0.000000
city                           -0.000344
fairy_lights                   -0.009239
drone_photographer             -0.018098
dhol_players                   -0.030648
event                          -0.034937
dtype: float64


In [18]:
X = X.drop(columns=["wedding_id"])

keep = ["aloo_count", "guests", "event", "fairy_lights",
        "drone_photographer", "city", "borhani_glasses"]

In [19]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_score(cols):
    num = [c for c in num_cols if c in cols]   # only numeric cols that are in `cols`
    cat = [c for c in cat_cols if c in cols]   # only text cols that are in `cols`

    pre = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])
    m = Pipeline([
        ("pre", pre),
        ("rf", RandomForestClassifier(n_estimators=300, random_state=42)),
    ])
    scores = cross_val_score(m, X[cols], y, cv=cv, scoring="f1_macro")
    return scores.mean(), scores.std()


all_cols = X.columns.tolist()

print("all columns:", cv_score(all_cols))
print("keep 7     :", cv_score(keep))

all columns: (np.float64(0.7570121014093398), np.float64(0.046420291591949714))
keep 7     : (np.float64(0.7655944020423585), np.float64(0.043598710765824646))


In [20]:
final_cols = ["aloo_count", "guests", "event", "fairy_lights",
              "drone_photographer", "city", "borhani_glasses"]

num = [c for c in num_cols if c in final_cols]
cat = [c for c in cat_cols if c in final_cols]

rf_model = Pipeline([
    ("pre", ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])),
    ("rf", RandomForestClassifier(n_estimators=500, random_state=42)),
])

rf_model.fit(X[final_cols], y)


Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['guests', 'aloo_count',
                                                   'borhani_glasses',
                                                   'fairy_lights']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['event', 'city',
                                                   'drone_photographer'])])),
                ('rf',
                 RandomForestClassifier(n_estimators=500, random_state=42))])

In [21]:
def add_features(d):
    d = d.copy()
    d["aloo_per_guest"] = d["aloo_count"] / d["guests"]
    d["mutton_per_guest"] = d["mutton_kg"] / d["guests"]
    d["lights_per_guest"] = d["fairy_lights"] / d["guests"]
    d["glasses_per_guest"] = d["borhani_glasses"] / d["guests"]
    return d

X2 = add_features(X)
test2 = add_features(test)

new = ["aloo_per_guest", "mutton_per_guest", "lights_per_guest", "glasses_per_guest"]
print(X2[new].describe())
print(test2[new].describe())

       aloo_per_guest  mutton_per_guest  lights_per_guest  glasses_per_guest
count      776.000000        776.000000        776.000000         776.000000
mean         1.537461          0.255410          2.819153           1.150360
std          0.674965          0.040471          1.243689           0.202087
min          0.406015          0.180308          0.560408           0.799652
25%          0.957247          0.220305          1.837893           0.968124
50%          1.506818          0.257545          2.714785           1.149611
75%          2.127857          0.290959          3.745286           1.321667
max          2.700000          0.319854          8.014706           1.496881
       aloo_per_guest  mutton_per_guest  lights_per_guest  glasses_per_guest
count      400.000000        400.000000        400.000000         400.000000
mean         1.529407          0.249229          2.668042           1.142611
std          0.640855          0.038437          1.161646           0.205251

In [22]:
cols = ["fairy_lights", "borhani_glasses"]

print(X2[cols].describe())
print(test[cols].describe())

print(X2[cols + ["guests"]].corr()["guests"])

       fairy_lights  borhani_glasses
count    776.000000       776.000000
mean     972.817010       428.715206
std      603.900507       231.128201
min       81.000000        45.000000
25%      511.500000       241.000000
50%      821.500000       411.000000
75%     1310.250000       598.250000
max     2861.000000      1016.000000
       fairy_lights  borhani_glasses
count    400.000000       400.000000
mean    1613.040000       739.775000
std     1286.793633       520.389953
min       98.000000        45.000000
25%      643.000000       321.500000
50%     1175.500000       592.500000
75%     2198.000000      1116.500000
max     5942.000000      2166.000000
fairy_lights       0.688641
borhani_glasses    0.929958
guests             1.000000
Name: guests, dtype: float64


In [23]:
#Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

new_cols = ["aloo_per_guest", "mutton_per_guest", "lights_per_guest", "glasses_per_guest", 
            "event", "drone_photographer", "city"]

num = [c for c in new_cols if c in X2.select_dtypes("number").columns]
cat = [c for c in new_cols if c not in num]

print("numeric:", num)
print("categorical:", cat)

lr_model = Pipeline([
    ("pre", ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])),
    ("clf", LogisticRegression(max_iter=1000)),
])

# shift split: train on smaller weddings, score on the biggest 20%
cut = X2["guests"].quantile(0.8)
small = X2["guests"] <= cut

lr_model.fit(X2.loc[small, new_cols], y[small])
pred = lr_model.predict(X2.loc[~small, new_cols])
print("shift F1:", f1_score(y[~small], pred, average="macro"))

# regular CV for comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("CV F1   :", cross_val_score(lr_model, X2[new_cols], y, cv=cv, scoring="f1_macro").mean())

numeric: ['aloo_per_guest', 'mutton_per_guest', 'lights_per_guest', 'glasses_per_guest']
categorical: ['event', 'drone_photographer', 'city']
shift F1: 0.4677815521296055
CV F1   : 0.4705169811352608


In [24]:
from sklearn.dummy import DummyClassifier

def shift_score(cols, model):
    num = [c for c in cols if c in X2.select_dtypes("number").columns]
    cat = [c for c in cols if c not in num]
    m = Pipeline([
        ("pre", ColumnTransformer([
            ("num", StandardScaler(), num),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
        ])),
        ("clf", model),
    ])
    m.fit(X2.loc[small, cols], y[small])
    pred = m.predict(X2.loc[~small, cols])
    return round(f1_score(y[~small], pred, average="macro"), 3)

old_cols = ["aloo_count", "guests", "event", "fairy_lights",
            "drone_photographer", "city", "borhani_glasses"]

rf = RandomForestClassifier(n_estimators=300, random_state=42)
lr = LogisticRegression(max_iter=1000)

print("baseline (all one class):", shift_score(new_cols, DummyClassifier(strategy="most_frequent")))
print("old cols  + RF:", shift_score(old_cols, rf))
print("new cols  + RF:", shift_score(new_cols, rf))
print("old cols  + LR:", shift_score(old_cols, lr))
print("new cols  + LR:", shift_score(new_cols, lr))
print("class balance :", y.value_counts(normalize=True).round(3).to_dict())

baseline (all one class): 0.356
old cols  + RF: 0.618
new cols  + RF: 0.954
old cols  + LR: 0.487
new cols  + LR: 0.468
class balance : {0: 0.577, 1: 0.423}


In [25]:
rf_cols = new_cols = ["aloo_per_guest", "mutton_per_guest", "lights_per_guest", "glasses_per_guest", 
            "event", "drone_photographer", "city"] 

num = [c for c in rf_cols if c in X2.select_dtypes("number").columns]
cat = [c for c in rf_cols if c not in num]

rf_model = Pipeline([
    ("pre", ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])),
    ("clf", RandomForestClassifier(n_estimators=1000, random_state=42)),
])

In [26]:
def add_features(d):
    d = d.copy()
    d["dhol_per_guest"]    = d["dhol_players"] / d["guests"]
    d["aunties_per_guest"] = d["aunties_asking_when_marriage"] / d["guests"]
    return d

X2 = add_features(X2)  
test2 = add_features(test2)

base  = ["aloo_per_guest", "mutton_per_guest", "lights_per_guest", "glasses_per_guest",
         "event", "drone_photographer", "city"]
extra = base + ["dhol_per_guest", "aunties_per_guest"]

rf = RandomForestClassifier(n_estimators=1000, random_state=42)
print("base :", shift_score(base, rf))
print("extra:", shift_score(extra, rf))

base : 0.96
extra: 0.96


In [27]:
rf_model.fit(X2[rf_cols], y)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['aloo_per_guest',
                                                   'mutton_per_guest',
                                                   'lights_per_guest',
                                                   'glasses_per_guest']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['event',
                                                   'drone_photographer',
                                                   'city'])])),
                ('clf',
                 RandomForestClassifier(n_estimators=1000, random_state=42))])

In [28]:
from sklearn.tree import DecisionTreeClassifier, export_text

feats = ["aloo_per_guest", "mutton_per_guest", "lights_per_guest", "glasses_per_guest"]

for depth in (1, 2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X2[feats], y)
    print("depth", depth, "accuracy:", round(t.score(X2[feats], y), 3))

t = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X2[feats], y)
print(export_text(t, feature_names=feats))

depth 1 accuracy: 0.706
depth 2 accuracy: 0.959
depth 3 accuracy: 0.961
depth 4 accuracy: 0.964
|--- aloo_per_guest <= 2.01
|   |--- aloo_per_guest <= 0.97
|   |   |--- aloo_per_guest <= 0.94
|   |   |   |--- class: 0
|   |   |--- aloo_per_guest >  0.94
|   |   |   |--- class: 0
|   |--- aloo_per_guest >  0.97
|   |   |--- aloo_per_guest <= 1.06
|   |   |   |--- class: 1
|   |   |--- aloo_per_guest >  1.06
|   |   |   |--- class: 1
|--- aloo_per_guest >  2.01
|   |--- aloo_per_guest <= 2.07
|   |   |--- mutton_per_guest <= 0.31
|   |   |   |--- class: 0
|   |   |--- mutton_per_guest >  0.31
|   |   |   |--- class: 1
|   |--- aloo_per_guest >  2.07
|   |   |--- mutton_per_guest <= 0.32
|   |   |   |--- class: 0
|   |   |--- mutton_per_guest >  0.32
|   |   |   |--- class: 0



In [29]:
r = X2["aloo_per_guest"]

for lo, hi in [(0.97, 2.01), (1.0, 2.0)]:
    pred = ((r > lo) & (r <= hi)).astype(int)
    print(lo, hi, "accuracy:", round((pred == y).mean(), 4))

0.97 2.01 accuracy: 0.9575
1.0 2.0 accuracy: 0.9459


In [30]:
pred = ((r > 1.0) & (r <= 2.0)).astype(int)
wrong = X2[pred != y]

print(len(wrong), "wrong out of", len(y))
print(wrong["aloo_per_guest"].sort_values().round(3).values)

42 wrong out of 776
[0.94  0.945 0.959 0.971 0.973 0.976 0.977 0.978 0.983 0.985 0.989 0.989
 1.034 1.042 1.043 1.047 1.051 1.058 1.152 1.684 1.74  1.804 1.824 1.835
 1.92  1.958 1.974 1.977 1.983 2.002 2.005 2.005 2.007 2.01  2.03  2.038
 2.053 2.058 2.068 2.121 2.205 2.253]


In [31]:
pd.DataFrame({
    "wedding_id": test2["wedding_id"],
    "went_back_for_seconds": rf_model.predict(test2[rf_cols]),
}).to_csv("submission.csv", index=False)